
# GigaPath 기반 WSI 분류 (mammary_adenoma vs mammary_adenocarcinoma)

- **대상**: curated parquet(`mammary_adenoma_vs_adenocarcinoma_curated.parquet`)의 `label` 컬럼을 사용해 양/악성 이진 분류.
- **경로 전략**: 외장(`/Volumes/Expansion/2023`)에서 슬라이드 존재 확인 → 균형 샘플링 → 레포 `Data/2023`으로 복사 후 로컬 경로로 타일/임베딩/학습.
- **모델**: HF Hub `prov-gigapath/prov-gigapath` 타일 인코더 + 단순 Attention MIL 헤드 예시.
- **실행 안내**: 셀을 순서대로 실행. GPU/드라이버, OpenSlide, pixman은 사전 설치되어 있다고 가정.


## 0. 필수 패키지 설치
- OpenSlide/pixman은 시스템 단에서 설치되어 있어야 합니다.
- timm, pandas, pyarrow, tqdm, ipywidgets 등을 설치합니다.


In [1]:

# 가상환경에 맞춰 필요 시 수정
%pip install --upgrade pip
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install 'timm>=1.0.8' pandas pyarrow scikit-learn scikit-image openslide-python matplotlib seaborn
%pip install huggingface_hub einops tqdm
%pip install ipywidgets


Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torchaudio (from versions: none)
ERROR: No matching distribution found for torchaudio
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 1. 경로/환경 설정
- 외장 슬라이드 경로(`/Volumes/Expansion/2023`)와 로컬 작업 경로(`Data/2023`)를 설정합니다.
- 캐시/타일/임베딩/체크포인트는 로컬 `Data/2023` 하위에 생성합니다.
- HF 토큰을 환경 변수 `HF_TOKEN`에 설정해야 모델 다운로드가 가능합니다.


In [2]:
import os
from pathlib import Path
import pandas as pd

# 원본 슬라이드 경로(외장)와 로컬 작업 경로
SOURCE_SLIDE_ROOT = Path('/Volumes/Expansion/2023')  # 예: '/Volumes/Expansion/2023'
DATA_ROOT = Path('/Users/curv/Repos/GC-Pathology/Data/2023')

# curated label parquet
PARQUET_PATH = Path('/Users/curv/Repos/GC-Pathology/PoC/v1/mammary_adenoma_vs_adenocarcinoma_curated.parquet')

# 캐시/타일/임베딩/체크포인트는 로컬 DATA_ROOT 하위에 생성
CACHE_ROOT = DATA_ROOT / 'gigapath_cache'
TILE_ROOT = CACHE_ROOT / 'tiles'
EMBED_ROOT = CACHE_ROOT / 'embeddings'
CKPT_ROOT = CACHE_ROOT / 'checkpoints'

# 초기 SLIDE_ROOT는 외장; 복사 후 로컬로 교체
SLIDE_ROOT = SOURCE_SLIDE_ROOT

if not SOURCE_SLIDE_ROOT.exists():
    raise FileNotFoundError(f'SOURCE_SLIDE_ROOT not found: {SOURCE_SLIDE_ROOT} (외장하드 경로 확인)')

for path in [DATA_ROOT, CACHE_ROOT, TILE_ROOT, EMBED_ROOT, CKPT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# HF 토큰은 .env(HF_TOKEN=...) 또는 환경변수로 세팅해 둡니다.


## 2. 라벨 데이터 로드 및 필터링 (curated parquet)
- `mammary_adenoma_vs_adenocarcinoma_curated.parquet`의 `label` 컬럼(`mammary_adenoma`/`mammary_adenocarcinoma`)을 사용합니다.
- 유효 라벨만 남기고 이진 라벨(0/1)로 매핑합니다.


In [3]:
df = pd.read_parquet(PARQUET_PATH)

label_norm = df['label'].astype(str).str.lower().str.strip()
valid = {'mammary_adenoma': 0, 'mammary_adenocarcinoma': 1}
filtered = df.loc[label_norm.isin(valid.keys())].copy()
filtered['label'] = label_norm.loc[filtered.index].map(valid)

print('전체 샘플 수:', len(filtered))
print(filtered[['INSP_RQST_NO', 'FOLDER', 'FILE_NAME', 'label']].head())


전체 샘플 수: 7835
         INSP_RQST_NO     FOLDER            FILE_NAME  label
0   20230101-117-0007  S23-00019  S23-00019#1###3.svs      0
1   20230101-117-0007  S23-00019  S23-00019#1###3.svs      0
6   20230101-127-0001  S23-00021  S23-00021#1###7.svs      0
7   20230101-127-0001  S23-00021  S23-00021#1###7.svs      0
13  20230102-136-0005  S23-00026  S23-00026#1###9.svs      0


## 3. 슬라이드 존재 + 균형 추출
- curated 라벨을 기준으로 pos/neg 최소 개수(cap, 최대 50)만큼 샘플링합니다.
- 외장 경로에서 실제 존재하는 슬라이드를 확인해 균형 잡힌 목록을 만듭니다.


In [4]:
# 라벨 기반 균형 샘플링 + 슬라이드 존재 확인
import random

# 라벨별 균형 맞춰 표본 추출 (cap=min(pos,neg), 최대 50)
pos_rows = filtered[filtered['label'] == 1].copy()
neg_rows = filtered[filtered['label'] == 0].copy()
if len(pos_rows) == 0 or len(neg_rows) == 0:
    print(f"원본 레이블 분포: pos={len(pos_rows)}, neg={len(neg_rows)}")
    raise ValueError('데이터에 두 클래스가 모두 있어야 합니다. parquet/필터를 확인하세요.')
cap = min(len(pos_rows), len(neg_rows), 50)
print(f'데이터 기반 cap: 각 클래스 {cap}개, 총 {cap*2}')

def collect(rows, need):
    rows = rows.sample(frac=1, random_state=42).to_dict('records')
    picked = []
    for row in rows:
        if len(picked) >= need:
            break
        slide_dir = SLIDE_ROOT / str(row['FOLDER']).strip()
        file_names = [fn.strip() for fn in str(row['FILE_NAME']).split('|') if fn.strip()]
        slide_id = row.get('INSP_RQST_NO', row.get('INSP_RQST_NUM', row['FOLDER']))
        for file_name in file_names:
            fn = file_name if file_name.lower().endswith('.svs') else f"{file_name}.svs"
            slide_path = slide_dir / fn
            if slide_path.exists():
                picked.append({'slide_id': slide_id, 'path': slide_path, 'label': int(row['label'])})
                break
    return picked

pos_pick = collect(pos_rows, cap)
neg_pick = collect(neg_rows, cap)
slide_paths = pos_pick + neg_pick
random.shuffle(slide_paths)

print(f'수집 결과: pos={len(pos_pick)}, neg={len(neg_pick)}, 총 {len(slide_paths)}')
if len(pos_pick) < cap or len(neg_pick) < cap:
    raise ValueError('필요한 슬라이드가 부족합니다. pos={}/{} , neg={}/{}'.format(len(pos_pick), cap, len(neg_pick), cap))

print(f'최종 밸런싱: pos={len(pos_pick)}, neg={len(neg_pick)} (총 {len(slide_paths)})')


데이터 기반 cap: 각 클래스 50개, 총 100
수집 결과: pos=50, neg=50, 총 100
최종 밸런싱: pos=50, neg=50 (총 100)


## 4. 선정 슬라이드 로컬 복사
- 외장 경로에서 선정된 슬라이드를 레포 `Data/2023/<FOLDER>/`로 복사합니다.
- 이후 작업은 로컬 `SLIDE_ROOT=Data/2023`을 사용합니다.


In [5]:
# 선정된 슬라이드 로컬 Data/2023으로 복사
import shutil
from tqdm import tqdm

DATA_ROOT.mkdir(parents=True, exist_ok=True)
local_slide_paths = []
for rec in tqdm(slide_paths, desc='copy to Data/2023'):
    src = rec['path']
    tgt_dir = DATA_ROOT / src.parent.name
    tgt_dir.mkdir(parents=True, exist_ok=True)
    tgt = tgt_dir / src.name
    if not tgt.exists():
        shutil.copy2(src, tgt)
    local_slide_paths.append({**rec, 'path': tgt})

slide_paths = local_slide_paths
SLIDE_ROOT = DATA_ROOT
print(f'로컬 복사 완료: {len(slide_paths)}개 -> {DATA_ROOT}')


copy to Data/2023: 100%|███████████████████| 100/100 [00:00<00:00, 22723.50it/s]

로컬 복사 완료: 100개 -> /Users/curv/Repos/GC-Pathology/Data/2023


## 5. 타일링 유틸리티 (메모리·디스크 안전 버전)
- 조직 마스크(Otsu threshold)로 배경을 제거하고 20×/256px 기준으로 타일링.
- 타일 PNG는 `TILE_ROOT`에 저장하고 메타를 남깁니다.


### 패치화 기준/흐름 메모 (상세)
- **언제/어디서 실행?** 4단계 로컬 복사 이후 `embed_slide`에서 `export_tiles` 호출 시 패치가 만들어집니다. 학습/추론 루프에서 임베딩 parquet(`Data/2023/gigapath_cache/embeddings/<slide_id>.parquet`)이 없을 때만 패치화가 다시 트리거되어 동일 슬라이드는 한 번만 타일링됩니다.
- **전체 흐름**:
  1. curated parquet 라벨 → 균형 샘플링 → 슬라이드를 `Data/2023/<FOLDER>/`로 복사.
  2. `embed_slide`가 없거나 parquet 캐시가 없으면 `export_tiles`로 PNG 패치를 생성(아래 분할/필터 기준 적용).
  3. `TileDataset`이 PNG → 텐서 변환 후 GigaPath tile encoder가 임베딩 parquet을 생성합니다. 이후 MIL 학습·검증·heatmap은 PNG를 다시 읽지 않고 parquet만 사용합니다.
- **분할/필터 기준**:
  - LEVEL=0(20×) 기준 패치 크기/stride=256px, 겹침 없음.
  - Otsu 기반 tissue mask에서 조직 비율 0.6 미만 패치는 배경으로 판단해 건너뜁니다.
  - 메타에 좌표(x, y), level을 그대로 남겨 재실행 시 동일 좌표가 유지됩니다(슬라이드별 일관된 heatmap/임베딩 가능).
- **산출물/캐시**: `Data/2023/gigapath_cache/tiles/<slide_id>/tile_{idx:06d}_x{x}_y{y}.png`와 메타 로그(`tile <파일명>: start ...`)가 남습니다. 이후 단계에서 parquet(`.../embeddings/<slide_id>.parquet`)만 사용하고 PNG는 필요 시 삭제 가능.
- **활용/검증 팁**: 로그의 `tile <파일명>: a/b tiles`로 예상 패치 수 대비 생성량을 확인하고, 동일 slide_id 재실행 시 로그가 뜨지 않으면 캐시가 정상 재사용된 것입니다.


In [6]:

import numpy as np
import openslide
from skimage import color, filters
from PIL import Image, ImageFile, PngImagePlugin
from tqdm import tqdm

# 일부 PNG 타일에서 큰 텍스트/ICC chunk로 인한 에러 방지
PngImagePlugin.MAX_TEXT_CHUNK = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

PATCH_SIZE = 256
MIN_TISSUE_RATIO = 0.6
LEVEL = 0  # 20x 레벨 가정; 필요 시 slide.level_dimensions 확인 후 조정

def tissue_mask(np_rgb: np.ndarray) -> np.ndarray:
    hsv = color.rgb2hsv(np_rgb)
    saturation = hsv[:, :, 1]
    thresh = filters.threshold_otsu(saturation)
    return saturation > thresh

def iter_tiles(slide: openslide.OpenSlide, stride: int = PATCH_SIZE):
    width, height = slide.level_dimensions[LEVEL]
    for y in range(0, height, stride):
        for x in range(0, width, stride):
            region = slide.read_region((x, y), LEVEL, (PATCH_SIZE, PATCH_SIZE)).convert('RGB')
            arr = np.array(region)
            mask = tissue_mask(arr)
            if mask.mean() < MIN_TISSUE_RATIO:
                continue
            yield x, y, region

def export_tiles(slide_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    slide = openslide.OpenSlide(str(slide_path))
    meta = []
    width, height = slide.level_dimensions[LEVEL]
    est_tiles = ((height + PATCH_SIZE - 1) // PATCH_SIZE) * ((width + PATCH_SIZE - 1) // PATCH_SIZE)
    total_tiles = est_tiles
    last_pct = -1
    print(f'tile {slide_path.name}: start ({total_tiles} tiles 예상)')
    print('  - 출력 예시: tile <파일명>: <진행%> (현재/총 타일 수)')
    for tile_idx, (x, y, tile_img) in enumerate(iter_tiles(slide)):
        pct = int((tile_idx + 1) * 100 / max(1, total_tiles))
        # tqdm 대신 1% 단위로만 로그를 찍어 콘솔 스팸을 줄입니다.
        if pct != last_pct:
        print(f'tile {slide_path.name}: {pct}% ({tile_idx+1}/{total_tiles} tiles)')
            last_pct = pct
        tile_name = f"tile_{tile_idx:06d}_x{x}_y{y}.png"
        tile_path = out_dir / tile_name
        tile_img.save(tile_path, format='PNG')
        meta.append({'tile_path': str(tile_path), 'x': x, 'y': y, 'level': LEVEL})
    slide.close()
    return meta


IndentationError: expected an indented block after 'if' statement on line 45 (369455085.py, line 46)

### 진행 로그 해석
- `tile <파일명>: start (N tiles 예상)` → 해당 WSI를 최대 N개 패치로 나눌 때의 시작 로그입니다.
- `tile <파일명>: P% (a/b tiles)` → 현재 WSI 타일 추출 진행률(P%), a는 현재까지 처리한 타일 수, b는 예상 총 타일 수입니다.
- train 로그의 `train epoch X` → 선택된 슬라이드 개수 기준으로 진행됩니다.


## 6. GigaPath 타일 인코더 로드
- HF Hub에서 모델을 다운로드합니다 (`HF_TOKEN` 필요, gated).
- timm으로 `prov-gigapath/prov-gigapath` 모델을 직접 로드해 타일 임베딩을 계산합니다.


In [7]:
import os
from pathlib import Path
import torch
import timm

if torch.cuda.is_available():
    DEVICE = 'cuda'
    AUTOCast_KWARGS = {'device_type': 'cuda', 'dtype': torch.float16}
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = 'mps'
    AUTOCast_KWARGS = None  # MPS autocast 지원 이슈 방지
else:
    DEVICE = 'cpu'
    AUTOCast_KWARGS = None

def _load_hf_token():
    # 우선 환경변수 확인
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if token:
        os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', token)
        return token
    # .env 후보 경로 탐색 (노트북 기준 ./, ../, ../../)
    candidates = [Path('.env'), Path('../.env'), Path('../../.env')]
    for path in candidates:
        if path.exists():
            for line in path.read_text().splitlines():
                if not line or line.strip().startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                if k.strip() in ('HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
                    token = v.strip()
                    os.environ['HF_TOKEN'] = token
                    os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', token)
                    return token
    raise RuntimeError('HF_TOKEN을 환경변수로 설정하거나 .env에 HF_TOKEN=... 값을 추가하세요 (gated 모델 접근 필요).')

def load_tile_encoder():
    """HF Hub에서 prov-gigapath tile encoder 로드 (timm).
    - 레포가 gated라 HF_TOKEN 또는 HUGGINGFACE_HUB_TOKEN이 필요합니다.
    """
    _load_hf_token()
    print('Downloading/Loading tile encoder from HF Hub (prov-gigapath/prov-gigapath)...')
    try:
        model = timm.create_model('hf_hub:prov-gigapath/prov-gigapath', pretrained=True)
    except Exception as e:
        raise RuntimeError('tile encoder 로드 실패: HF 네트워크/토큰을 확인하세요.') from e
    return model.eval().to(DEVICE)

encoder = load_tile_encoder()


Downloading/Loading tile encoder from HF Hub (prov-gigapath/prov-gigapath)...


## 7. 타일 임베딩 추출 및 캐싱
- 배치 단위로 타일을 로드하여 FP16 추론 후 parquet/h5로 저장.
- 스토리지 절약을 위해 float16로 저장.


In [8]:
# 슬라이드 타일 추출 후 임베딩 계산
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pyarrow as pa
import pyarrow.parquet as pq
from contextlib import nullcontext
from tqdm import tqdm

transform = transforms.Compose([
    transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class TileDataset(Dataset):
    def __init__(self, tile_meta):
        self.tile_meta = tile_meta
    def __len__(self):
        return len(self.tile_meta)
    def __getitem__(self, idx):
        entry = self.tile_meta[idx]
        img = Image.open(entry['tile_path']).convert('RGB')
        return transform(img), entry['x'], entry['y']

@torch.inference_mode()
def embed_slide(slide_rec):
    slide_id = slide_rec['slide_id']
    print(f'embed {slide_id}: start (tiles will be generated if absent)')
    slide_path = slide_rec['path']
    out_dir = TILE_ROOT / str(slide_id)
    tile_meta = export_tiles(slide_path, out_dir)

    if not tile_meta:
        raise ValueError(f'No tiles extracted from {slide_path}')

    ds = TileDataset(tile_meta)
    # 워커 크래시/디버그를 위해 멀티프로세싱 대신 단일 스레드 로딩
    loader = DataLoader(ds, batch_size=64, num_workers=0, pin_memory=(DEVICE == 'cuda'))

    coords, feats = [], []
    autocast_ctx = torch.autocast(**AUTOCast_KWARGS) if AUTOCast_KWARGS else nullcontext()
    for imgs, xs, ys in tqdm(loader, desc=f"tile->embed {slide_id}"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast_ctx:
            emb = encoder(imgs)
        feats.append(emb.detach().to('cpu', dtype=torch.float16))
        coords.extend(zip(xs.tolist(), ys.tolist()))

    feat_tensor = torch.cat(feats, dim=0)
    table = pa.Table.from_pydict({
        'x': [c[0] for c in coords],
        'y': [c[1] for c in coords],
        'feature': feat_tensor.numpy().tolist(),
    })

    out_path = EMBED_ROOT / f"{slide_id}.parquet"
    pq.write_table(table, out_path, compression='zstd')
    return out_path


## 8. 간단한 MIL 헤드(Attention) 학습 예시
- 개념 증명용 소형 Attention MIL 헤드를 정의합니다.
- 임베딩을 메모리/디스크에서 적재해 학습합니다.


In [9]:

import math
import torch.nn as nn

class AttentionMIL(nn.Module):
    def __init__(self, in_dim, hidden_dim=256):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.classifier = nn.Linear(in_dim, 1)
    def forward(self, feats):
        attn_score = self.attn(feats)  # (N,1)
        weight = torch.softmax(attn_score, dim=0)
        slide_feat = (weight * feats).sum(dim=0, keepdim=True)
        logit = self.classifier(slide_feat)
        return logit.squeeze(0), weight.squeeze(1)


## 9. 학습 루프 스켈레톤
- 균형 잡힌 슬라이드 리스트로 train/val을 구성합니다.
- 필요 시 임베딩이 없으면 on-the-fly로 생성합니다.


In [ ]:
# Attention MIL 헤드 학습 루프

import random
from sklearn.model_selection import train_test_split
import torch.optim as optim
from tqdm import tqdm

# 소규모 데이터에서 stratify가 단일 클래스로 쏠리는 이슈 방지: 수동 분할
by_label = {0: [], 1: []}
for rec in slide_paths:
    by_label[int(rec['label'])].append(rec)
for lst in by_label.values():
    random.shuffle(lst)

if not by_label[0] or not by_label[1]:
    print('경고: 데이터가 단일 클래스입니다. 모든 슬라이드를 train에 사용하고 val은 비웁니다.')
    train_recs = slide_paths.copy()
    val_recs = []
else:
    val_recs, train_recs = [], []
    for lbl, lst in by_label.items():
        take = 1 if len(lst) > 1 else 0  # 각 클래스에서 최소 1개를 val로 (여유 있을 때만)
        val_recs.extend(lst[:take])
        train_recs.extend(lst[take:])

# 클래스 비율 확인 (0=adenoma, 1=adenocarcinoma)
train_pos = sum(r['label'] for r in train_recs)
train_neg = len(train_recs) - train_pos
val_pos = sum(r['label'] for r in val_recs)
val_neg = len(val_recs) - val_pos
print(f"train count: pos={train_pos}, neg={train_neg}")
print(f"val   count: pos={val_pos}, neg={val_neg}")

# timm 비전 모델은 embedding 차원을 num_features로 노출 (output_dim이 없을 수 있음)
embed_dim = getattr(encoder, 'output_dim', None) or getattr(encoder, 'num_features', None)
if embed_dim is None:
    raise AttributeError('encoder의 embedding dim을 찾을 수 없습니다 (output_dim/num_features).')

model = AttentionMIL(in_dim=embed_dim).to(DEVICE)
if train_pos == 0 or train_neg == 0:
    pos_weight = torch.tensor([1.0], device=DEVICE)
else:
    pos_weight = torch.tensor([(train_neg / train_pos)], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 5

@torch.inference_mode()
def load_embed(path: Path):
    table = pq.read_table(path)
    feats = torch.tensor(np.stack(table['feature'].to_numpy()), dtype=torch.float16)
    coords = np.stack([table['x'].to_numpy(), table['y'].to_numpy()], axis=1)
    return feats, coords

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_recs)
    total_loss = 0
    for rec in tqdm(train_recs, desc=f"train epoch {epoch}"):
        embed_path = EMBED_ROOT / f"{rec['slide_id']}.parquet"
        if not embed_path.exists():
            embed_path = embed_slide(rec)
        feats, _ = load_embed(embed_path)
        feats = feats.to(DEVICE)

        logit, _ = model(feats)
        label = torch.tensor([rec['label']], device=DEVICE, dtype=torch.float32)
        loss = criterion(logit, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch}: train_loss={total_loss/len(train_recs):.4f}")

    if val_recs:
        model.eval()
        correct = 0
        with torch.no_grad():
            for rec in tqdm(val_recs, desc=f"val epoch {epoch}"):
                embed_path = EMBED_ROOT / f"{rec['slide_id']}.parquet"
                if not embed_path.exists():
                    embed_path = embed_slide(rec)
                feats, _ = load_embed(embed_path)
                feats = feats.to(DEVICE)
                logit, _ = model(feats)
                pred = (torch.sigmoid(logit) > 0.5).int().item()
                correct += int(pred == rec['label'])
        acc = correct / len(val_recs) if val_recs else float('nan')
        print(f"Val acc: {acc:.3f}")
    else:
        print('val set 없음 (단일 클래스 또는 극소 샘플)')

torch.save(model.state_dict(), CKPT_ROOT / 'attention_mil.pt')


train count: pos=49, neg=49
val   count: pos=1, neg=1


tile S23-07617#1###0.svs:   2%|▏            | 357/23048 [00:25<24:01, 15.75it/s]

## 10. 슬라이드 단위 추론 + 패치 중요도 heatmap 초안
- attention weight를 이용해 패치 좌표별 중요도를 얻습니다.
- 추후 썸네일과 overlay 가능하도록 좌표/score를 parquet로 저장합니다.


In [ ]:

import matplotlib.pyplot as plt

@torch.inference_mode()
def predict_slide(rec):
    embed_path = EMBED_ROOT / f"{rec['slide_id']}.parquet"
    if not embed_path.exists():
        embed_path = embed_slide(rec)
    feats, coords = load_embed(embed_path)
    feats = feats.to(DEVICE)
    logit, attn = model(feats)
    prob = torch.sigmoid(logit).item()
    return prob, coords, attn.cpu().numpy()

example = train_recs[0]
prob, coords, attn = predict_slide(example)
print(f"Slide {example['slide_id']} prob(malignant)={prob:.3f}")

plt.figure(figsize=(6, 5))
plt.scatter(coords[:,0], coords[:,1], c=attn, s=10, cmap='hot')
plt.gca().invert_yaxis()
plt.colorbar(label='Attention')
plt.title('Patch importance (higher = more malignant)')
plt.show()


## 11. 후처리/청소
- 타일 PNG를 더 이상 사용하지 않는다면 용량 절약을 위해 삭제하세요.
- parquet 임베딩은 재사용을 위해 남겨둡니다.


In [ ]:

import shutil

REMOVE_TILES = False
if REMOVE_TILES:
    shutil.rmtree(TILE_ROOT, ignore_errors=True)
